> **Disclaimer:** This is a research project. Models are not validated
> for clinical use and must not be used for medical diagnosis or
> treatment decisions. The NIH ChestX-ray14 labels were text-mined from
> radiology reports with an estimated 10–15% label noise.

# Model Analysis and Interpretability

**Purpose:** Deep dive analysis of best performing model

**Analyses included:**
1. Error analysis - identify failure modes
2. Confidence calibration
3. Grad-CAM visualization - model attention
4. Confusion analysis
5. Prediction examples

## Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import cv2
from PIL import Image
import json
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.calibration import calibration_curve

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
from src.dataset import ChestXrayDataset, apply_clahe, get_transforms


## Configuration

In [ ]:
DATA_ROOT = Path('./data')
PROCESSED_DIR = DATA_ROOT / 'processed'
MODELS_DIR = Path('./models')
ANALYSIS_DIR = Path('./analysis')
ANALYSIS_DIR.mkdir(exist_ok=True)

BATCH_SIZE = 16
NUM_WORKERS = 4

RESULTS_FILE = Path('./results/test_results.json')
if not RESULTS_FILE.exists():
    raise FileNotFoundError(
        f"Results file not found: {RESULTS_FILE}
"
        "Run 07_evaluate_models.ipynb first to generate test_results.json."
    )

import json as _json
with open(RESULTS_FILE) as _f:
    _eval_results = _json.load(_f)

MODEL_NAME = _eval_results['best_model']
MODEL_PATH = MODELS_DIR / MODEL_NAME / 'best_model.pth'
print(f"Best model from evaluation: {MODEL_NAME}")

## Load Data

In [ ]:
test_df = pd.read_csv(PROCESSED_DIR / 'test_df.csv')

with open(PROCESSED_DIR / 'preprocessing_config.json', 'r') as f:
    config = json.load(f)

diseases = config['diseases']
NUM_CLASSES = len(diseases)
IMG_SIZE = config['image_size']

print(f"Test samples: {len(test_df):,}")
print(f"Diseases: {diseases}")

## Dataset

In [ ]:
# Moved to src/dataset.py

In [ ]:
_, val_transform = get_transforms(IMG_SIZE)
test_transform = val_transform

test_dataset = ChestXrayDataset(
    test_df, diseases, transform=test_transform, return_path=True
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

## Load Model

In [ ]:
class EfficientNetB0Model(nn.Module):
    def __init__(self, num_classes):
        super(EfficientNetB0Model, self).__init__()
        self.backbone = models.efficientnet_b0(weights=None)
        self.backbone.features[0][0] = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)
        num_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_features, num_classes)
        )
    def forward(self, x):
        return self.backbone(x)

model = EfficientNetB0Model(NUM_CLASSES)
# weights_only=True is safe — checkpoint contains only tensors and strings
checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=True)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

print(f"Model loaded: {MODEL_NAME}")

## Get Predictions

In [ ]:
all_labels = []
all_preds = []
all_paths = []

with torch.no_grad():
    for images, labels, paths in tqdm(test_loader, desc='Getting predictions'):
        images = images.to(device)
        outputs = model(images)
        preds = torch.sigmoid(outputs)
        
        all_labels.append(labels.cpu().numpy())
        all_preds.append(preds.cpu().numpy())
        all_paths.extend(paths)

all_labels = np.vstack(all_labels)
all_preds = np.vstack(all_preds)

print(f"Predictions shape: {all_preds.shape}")

## Error Analysis

### Identify High-Confidence Errors

In [ ]:
threshold = 0.5
errors_data = []

for i, disease in enumerate(diseases):
    y_true = all_labels[:, i]
    y_pred = all_preds[:, i]
    y_pred_binary = (y_pred > threshold).astype(int)
    
    false_positives = (y_true == 0) & (y_pred_binary == 1)
    false_negatives = (y_true == 1) & (y_pred_binary == 0)
    
    fp_indices = np.where(false_positives)[0]
    fn_indices = np.where(false_negatives)[0]
    
    if len(fp_indices) > 0:
        high_conf_fp = fp_indices[np.argsort(y_pred[fp_indices])[-5:]]
        for idx in high_conf_fp:
            errors_data.append({
                'disease': disease,
                'error_type': 'False Positive',
                'confidence': y_pred[idx],
                'image_path': all_paths[idx]
            })
    
    if len(fn_indices) > 0:
        high_conf_fn = fn_indices[np.argsort(y_pred[fn_indices])[:5]]
        for idx in high_conf_fn:
            errors_data.append({
                'disease': disease,
                'error_type': 'False Negative',
                'confidence': y_pred[idx],
                'image_path': all_paths[idx]
            })

errors_df = pd.DataFrame(errors_data)
errors_df.to_csv(ANALYSIS_DIR / 'high_confidence_errors.csv', index=False)

print(f"High confidence errors: {len(errors_df)}")
print(f"\nError distribution:")
print(errors_df['error_type'].value_counts())

### Error Rate by Disease

In [ ]:
error_rates = []

for i, disease in enumerate(diseases):
    y_true = all_labels[:, i]
    y_pred = (all_preds[:, i] > threshold).astype(int)
    
    total = len(y_true)
    errors = np.sum(y_true != y_pred)
    error_rate = errors / total
    
    error_rates.append({
        'Disease': disease,
        'Error Rate': error_rate,
        'Total Samples': total,
        'Errors': errors
    })

error_rates_df = pd.DataFrame(error_rates).sort_values('Error Rate', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(error_rates_df['Disease'], error_rates_df['Error Rate'], color='coral')
ax.set_xlabel('Error Rate')
ax.set_title('Error Rate by Disease')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'error_rates.png', dpi=300, bbox_inches='tight')
plt.show()

print(error_rates_df.to_string(index=False))

## Confidence Calibration

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for idx, disease in enumerate(diseases[:6]):
    i = diseases.index(disease)
    y_true = all_labels[:, i]
    y_pred = all_preds[:, i]
    
    if len(np.unique(y_true)) > 1:
        fraction_of_positives, mean_predicted_value = calibration_curve(
            y_true, y_pred, n_bins=10, strategy='uniform'
        )
        
        ax = axes[idx]
        ax.plot(mean_predicted_value, fraction_of_positives, 's-', label=disease)
        ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
        ax.set_xlabel('Mean Predicted Probability')
        ax.set_ylabel('Fraction of Positives')
        ax.set_title(disease)
        ax.legend(loc='lower right')
        ax.grid(True, alpha=0.3)

plt.suptitle('Calibration Curves', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'calibration_curves.png', dpi=300, bbox_inches='tight')
plt.show()

## Grad-CAM Visualization

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        self.target_layer.register_forward_hook(self.save_activation)
        self.target_layer.register_backward_hook(self.save_gradient)
    
    def save_activation(self, module, input, output):
        self.activations = output.detach()
    
    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()
    
    def generate_cam(self, input_image, target_class):
        self.model.zero_grad()
        output = self.model(input_image)
        
        target = output[:, target_class]
        target.backward()
        
        gradients = self.gradients[0]
        activations = self.activations[0]
        
        weights = torch.mean(gradients, dim=(1, 2))
        cam = torch.zeros(activations.shape[1:], dtype=torch.float32)
        
        for i, w in enumerate(weights):
            cam += w * activations[i]
        
        cam = F.relu(cam)
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        
        return cam.cpu().numpy()

target_layer = model.backbone.features[-1]
grad_cam = GradCAM(model, target_layer)

In [ ]:
def visualize_gradcam(image_path, disease_idx, disease_name, save_name):
    image = Image.open(image_path).convert('L')
    original = np.array(image)
    
    image_tensor = test_transform(image).unsqueeze(0).to(device)
    image_tensor.requires_grad = True
    
    cam = grad_cam.generate_cam(image_tensor, disease_idx)
    cam_resized = cv2.resize(cam, (original.shape[1], original.shape[0]))
    
    heatmap = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    
    original_rgb = cv2.cvtColor(original, cv2.COLOR_GRAY2RGB)
    overlay = cv2.addWeighted(original_rgb, 0.6, heatmap, 0.4, 0)
    
    with torch.no_grad():
        output = model(image_tensor)
        prob = torch.sigmoid(output)[0, disease_idx].item()
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(original, cmap='gray')
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    axes[1].imshow(cam_resized, cmap='jet')
    axes[1].set_title('Grad-CAM Heatmap')
    axes[1].axis('off')
    
    axes[2].imshow(overlay)
    axes[2].set_title(f'{disease_name}\nPrediction: {prob:.3f}')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.savefig(ANALYSIS_DIR / f'gradcam_{save_name}.png', dpi=300, bbox_inches='tight')
    plt.show()

### Generate Grad-CAM for Sample Cases

In [ ]:
for disease_idx, disease in enumerate(diseases[:3]):
    positive_cases = np.where(all_labels[:, disease_idx] == 1)[0]
    if len(positive_cases) > 0:
        high_conf_idx = positive_cases[np.argmax(all_preds[positive_cases, disease_idx])]
        image_path = all_paths[high_conf_idx]
        
        print(f"\nGenerating Grad-CAM for {disease}...")
        visualize_gradcam(
            image_path,
            disease_idx,
            disease,
            disease.lower().replace(' ', '_')
        )

## Confusion Analysis

In [ ]:
co_occurrence = np.zeros((NUM_CLASSES, NUM_CLASSES))

for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        both_positive = np.sum((all_labels[:, i] == 1) & (all_labels[:, j] == 1))
        co_occurrence[i, j] = both_positive

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    co_occurrence,
    annot=True,
    fmt='.0f',
    cmap='Blues',
    xticklabels=diseases,
    yticklabels=diseases,
    ax=ax
)
ax.set_title('Disease Co-occurrence Matrix (Test Set)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'disease_cooccurrence.png', dpi=300, bbox_inches='tight')
plt.show()

## Prediction Distribution

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.ravel()

for i, disease in enumerate(diseases):
    if i < len(axes):
        ax = axes[i]
        
        positive_preds = all_preds[all_labels[:, i] == 1, i]
        negative_preds = all_preds[all_labels[:, i] == 0, i]
        
        ax.hist(negative_preds, bins=50, alpha=0.5, label='Negative', color='blue')
        ax.hist(positive_preds, bins=50, alpha=0.5, label='Positive', color='red')
        ax.axvline(x=0.5, color='black', linestyle='--', linewidth=1)
        ax.set_xlabel('Predicted Probability')
        ax.set_ylabel('Count')
        ax.set_title(disease)
        ax.legend()

if len(diseases) < len(axes):
    axes[-1].axis('off')

plt.suptitle('Prediction Distributions by Disease', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'prediction_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

## Summary Report

In [ ]:
summary = {
    'model': MODEL_NAME,
    'test_set_size': len(test_df),
    'diseases': diseases,
    'performance': {},
    'error_analysis': {}
}

for i, disease in enumerate(diseases):
    y_true = all_labels[:, i]
    y_pred = all_preds[:, i]
    
    if len(np.unique(y_true)) > 1:
        auc = roc_auc_score(y_true, y_pred)
    else:
        auc = 0.0
    
    y_pred_binary = (y_pred > 0.5).astype(int)
    errors = np.sum(y_true != y_pred_binary)
    error_rate = errors / len(y_true)
    
    summary['performance'][disease] = {
        'auroc': float(auc),
        'error_rate': float(error_rate),
        'total_errors': int(errors)
    }

with open(ANALYSIS_DIR / 'analysis_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("Analysis complete!")
print(f"Results saved to {ANALYSIS_DIR}")